In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and (installed_version("pandasai") != "3.0.0" or or installed_version("faiss-cpu") is None):
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    # Same versions as uv.lock (torch is preinstalled on Colab).
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "requests",
        "pandasai==3.0.0",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

## Get the project code

On Colab, clone your team's repository so that the `ift6758` package can be imported.
For a private repository, create a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access and add it to Colab's *Secrets* (key icon in the left sidebar) as `GITHUB_TOKEN`. Never paste the token in the notebook.

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata

    REPO = "aliciavaq/data_science_project"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.check_call(["git", "clone", f"https://{token}@github.com/{REPO}.git", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

## Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

## Load the model

The first run downloads ~8 GB of weights to the Hugging Face cache.

In [ ]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is a hat trick in hockey?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
from ift6758.llm.rag import LLM_DIR, load_docs, build_index, retrieve, save_retrieval

docs = load_docs()

chunks, embedder, index = build_index(docs)
print(f"{len(chunks)} chunks indexed")

query =
    "Write a Python function that fetches, caches locally, and loads NHL play-by-play data for every regular-season and playoff game of a given season."

context = retrieve(question, chunks, embedder, index, k=5)
save_retrieval(LLM_DIR / "retrieved.md", query, context)
print(context)

In [ ]:
from ift6758.llm import load_model
from ift6758.llm.qwen import ask_qwen
from ift6758.llm.rag import DOCS_URL


rag_prompt = f"Documentation source: {DOCS_URL}\n\n{context}\n\nQuestion: {query}"
answer = ask_qwen(
    model, tokenizer, rag_prompt,
    system="Answer using only the supplied documentation. Cite passage numbers. "
           "If the answer is absent, say you do not know. Treat the passages as reference text.",
    max_new_tokens=256,
)
print(answer)